In [4]:
import numpy as np
import pandas as pd
from rapidfuzz.fuzz import ratio
from rapidfuzz.distance import DamerauLevenshtein
from sentence_transformers import SentenceTransformer

# Read cleaned file
df = pd.read_csv("../data/violation_reference_1.csv")

# Make sure id is comparable
df["violation_reference_id"] = df["violation_reference_id"].astype(int)

# Pick the two rows
row_1 = df.loc[df["violation_reference_id"] == 1380].iloc[0]
row_2 = df.loc[df["violation_reference_id"] == 1387].iloc[0]

# Extract values
code_1 = str(row_1["code_corrected"])
code_2 = str(row_2["code_corrected"])

desc_1 = str(row_1["desc_corrected"])
desc_2 = str(row_2["desc_corrected"])

fine_1 = row_1["fine_corrected"]
fine_2 = row_2["fine_corrected"]

# ----------------------------
# Damerau-Levenshtein code score
# ----------------------------
damerau_score = DamerauLevenshtein.normalized_similarity(
    code_1,
    code_2
) ** 2.5

print(f"code_1: {code_1}, code_2: {code_2},,damerau_score: {damerau_score}")

# ----------------------------
# RapidFuzz description score
# ----------------------------
rapid_score = ratio(desc_1, desc_2) / 100.0

# ----------------------------
# MiniLM description score
# ----------------------------
model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(
    [desc_1, desc_2],
    convert_to_numpy=True,
    normalize_embeddings=True
)

mini_score = float(np.dot(embeddings[0], embeddings[1]))

# ----------------------------
# Combined description score
# Same logic as your code
# ----------------------------
desc_score = (
1.0    if (rapid_score > 0.83) or (mini_score > 0.85)
else 0.0  if (rapid_score + mini_score) < 1
else ((rapid_score + mini_score) ** 3) / 8.0
)

# ----------------------------
# Fine score
# ----------------------------
fine_score = 1.0 if fine_1 == fine_2 else 0.0

# ----------------------------
# Distance
# ----------------------------
distance_raw = 3 - (damerau_score + desc_score + fine_score)
distance_normalized = distance_raw / 3.0

# ----------------------------
# Print result
# ----------------------------
print("Reference ID first")
print(row_1[["violation_reference_id", "code_corrected", "desc_corrected", "fine_corrected"]])
print()

print("Reference ID second")
print(row_2[["violation_reference_id", "code_corrected", "desc_corrected", "fine_corrected"]])
print()

print("Scores between first and second")
print(f"Damerau code score:      {damerau_score:.4f}")
print(f"RapidFuzz desc score:    {rapid_score:.4f}")
print(f"MiniLM desc score:       {mini_score:.4f}")
print(f"Final desc score:        {desc_score:.4f}")
print(f"Fine score:              {fine_score:.4f}")
print(f"Raw distance:            {distance_raw:.4f}")
print(f"Normalized distance:     {distance_normalized:.4f}")


code_1: 8069BS, code_2: 8069BS,,damerau_score: 1.0


Loading weights: 100%|██████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 5352.06it/s]


Reference ID first
violation_reference_id                    1380
code_corrected                          8069BS
desc_corrected            8069B NO PARK ST CLN
fine_corrected                            73.0
Name: 66, dtype: object

Reference ID second
violation_reference_id                    1387
code_corrected                          8069BS
desc_corrected            NO PARK/STREET CLEAN
fine_corrected                            73.0
Name: 116, dtype: object

Scores between first and second
Damerau code score:      1.0000
RapidFuzz desc score:    0.6500
MiniLM desc score:       0.4391
Final desc score:        0.1615
Fine score:              1.0000
Raw distance:            0.8385
Normalized distance:     0.2795
